# DEEPX Tutorial 10 — PP-OCRv6 on DEEPX NPU

This tutorial builds a complete OCR pipeline with PP-OCRv5 models and a DEEPX NPU.

The workflow is divided into stages that can be checked independently:

1. download the original ONNX models;
2. replace dynamic input dimensions with fixed shapes;
3. verify that the fixed models preserve the ONNX results;
4. compile the models to DXNN;
5. run detection, orientation classification, and text recognition as one application.

> The original <code>paddleocr.ipynb</code> is kept unchanged. This reviewed notebook uses separate configuration and output directories.

## Learning objectives

After completing this tutorial, you will be able to:

- explain the detection → orientation classification → recognition OCR pipeline;
- explain why one dynamic recognition model becomes six fixed-shape DXNN models;
- safely download and validate the source ONNX files;
- create calibration configurations whose preprocessing matches the application;
- compile models with <code>dxcom</code> without hiding failures;
- inspect the generated DXNN artifacts;
- run the reviewed camera application and identify practical OCR accuracy limits.

### Review scope

This v6 review includes source-code inspection, model URL checks, ONNX validation, fixed-shape conversion, representative ONNX Runtime equivalence tests, and a DX-COM smoke compile. The complete camera pipeline still requires a DEEPX NPU, the target SDK installation, and a camera; those hardware-dependent checks are clearly separated in Section 7.

## 1. Understand the OCR pipeline

PP-OCRv5 uses three model stages.

| Stage | Input | Output | Purpose |
|---|---|---|---|
| Text detection | Full image, 480 × 480 | Text polygons | Find text regions |
| Orientation classification | One text crop, 48 × 192 | 0° or 180° | Correct upside-down crops |
| Text recognition | One text crop, height 48 | Character sequence | Convert pixels to text |

<img src="assets/ocr-workflow.jpg" style="max-width: 980px;" alt="PP-OCR workflow">

A single frame can contain many text regions. Detection runs once per frame, while classification and recognition run once for each detected crop.

### 1.1 Why recognition uses six models

The original recognition ONNX model accepts a dynamic width. A compiled NPU model needs a fixed input shape, so this tutorial creates six width buckets.

| Router bucket | NCHW input | Maximum width-to-height ratio |
|---:|---:|---:|
| 3 | [1, 3, 48, 120] | 2.5 |
| 5 | [1, 3, 48, 240] | 5 |
| 10 | [1, 3, 48, 480] | 10 |
| 15 | [1, 3, 48, 720] | 15 |
| 25 | [1, 3, 48, 1200] | 25 |
| 35 | [1, 3, 48, 1920] | 40 |

The bucket name is a routing label. The exact maximum ratio is <code>width / 48</code>; therefore, the first and last labels are not exact mathematical ratios.

<img src="assets/ocr-ratio.png" style="max-width: 900px;" alt="OCR recognition ratio routing">

## 2. Prepare the tutorial environment

This notebook reads <code>config.json</code> through the shared tutorial path helper. It does not assume that the SDK is installed inside the tutorial repository.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError(
        "ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh."
    )

TUTORIAL_ROOT = Path(root_path).expanduser().resolve()
sys.path.insert(0, str(TUTORIAL_ROOT))

from tutorial_paths import load_tutorial_path_vars, print_tutorial_paths

globals().update(load_tutorial_path_vars(TUTORIAL_ROOT))
print_tutorial_paths(TUTORIAL_ROOT)

TUTORIAL_DIR = TUTORIAL_ROOT / "notebooks" / "T10-PaddleOCRv5"
MODEL_DIR = TUTORIAL_DIR / "models"
CONFIG_DIR = TUTORIAL_DIR / "configs_v6"
OUTPUT_DIR = TUTORIAL_DIR / "outputs" / "paddleocr_v6"

DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"
DXCOM_PATH = DX_COMPILER_VENV / "bin" / "dxcom"
DXPARSE_PATH = Path("/usr/local/bin/dxparse")
DX_ENGINE_PACKAGE_DIR = DX_RT_DIR / "python_package"

for directory in (MODEL_DIR, CONFIG_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

for required in (TUTORIAL_DIR, DXCOM_PATH, DX_ENGINE_PACKAGE_DIR):
    if not required.exists():
        raise FileNotFoundError(f"Required path was not found: {required}")

print(f"Tutorial directory : {TUTORIAL_DIR}")
print(f"Model directory    : {MODEL_DIR}")
print(f"Config directory   : {CONFIG_DIR}")
print(f"Output directory   : {OUTPUT_DIR}")

### 2.1 Install Python packages into the current uv environment

The Jupyter environment was created with uv and may not contain the <code>pip</code> module. The next cell therefore uses <code>uv pip install --python ...</code> instead of <code>%pip</code>.

Re-running the cell is safe: uv reuses packages that already satisfy the requirements.

In [ ]:
UV_PATH = shutil.which("uv")
if UV_PATH is None:
    raise FileNotFoundError(
        "uv is not installed. Install uv, then restart JupyterLab with ./run-jupyter-lab.sh."
    )

NOTEBOOK_PACKAGES = [
    "onnx",
    "onnxruntime",
    "onnxsim",
    "opencv-python",
    "pillow",
    "pyclipper",
    "shapely",
    "torch",
]

install_command = [
    UV_PATH, "pip", "install",
    "--python", sys.executable,
    *NOTEBOOK_PACKAGES,
]
print("Equivalent terminal command:")
print(" ".join(map(str, install_command)))
subprocess.run(install_command, check=True)

## 3. Download and inspect the ONNX models

Downloads use HTTPS certificate verification. Each file is first written with a <code>.part</code> suffix and is renamed only after a complete transfer. A non-empty existing file is reused.

In [ ]:
from urllib.request import Request, urlopen

MODEL_URLS = {
    "det.onnx": (
        "https://github.com/jingsongliujing/OnnxOCR/raw/refs/heads/main/"
        "onnxocr/models/ppocrv5/det/ppocrv5_mobile_det.onnx"
    ),
    "cls.onnx": (
        "https://github.com/jingsongliujing/OnnxOCR/raw/refs/heads/main/"
        "onnxocr/models/ppocrv5/cls/ppocrv5_mobile_cls.onnx"
    ),
    "rec.onnx": (
        "https://github.com/jingsongliujing/OnnxOCR/raw/refs/heads/main/"
        "onnxocr/models/ppocrv5/rec/ppocrv5_mobile_rec.onnx"
    ),
}

def download_file(url: str, destination: Path) -> Path:
    if destination.is_file() and destination.stat().st_size > 0:
        print(f"SKIP: {destination.name} already exists "
              f"({destination.stat().st_size / 1_000_000:.1f} MB)")
        return destination

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)

    request = Request(url, headers={"User-Agent": "dx-tutorials"})
    try:
        with urlopen(request, timeout=120) as response, temporary.open("wb") as output:
            expected = response.headers.get("Content-Length")
            shutil.copyfileobj(response, output)
        if temporary.stat().st_size == 0:
            raise RuntimeError(f"Downloaded file is empty: {destination.name}")
        if expected and temporary.stat().st_size != int(expected):
            raise RuntimeError(
                f"Incomplete download for {destination.name}: "
                f"{temporary.stat().st_size} of {expected} bytes"
            )
        temporary.replace(destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise

    print(f"DOWNLOADED: {destination.name} "
          f"({destination.stat().st_size / 1_000_000:.1f} MB)")
    return destination

ONNX_MODELS = {
    name: download_file(url, MODEL_DIR / name)
    for name, url in MODEL_URLS.items()
}

In [ ]:
import onnx

def tensor_shape(value_info):
    return [
        dim.dim_value if dim.HasField("dim_value") else dim.dim_param or "dynamic"
        for dim in value_info.type.tensor_type.shape.dim
    ]

for name, model_path in ONNX_MODELS.items():
    onnx.checker.check_model(str(model_path))
    model = onnx.load(model_path, load_external_data=False)
    inputs = [(value.name, tensor_shape(value)) for value in model.graph.input]
    outputs = [(value.name, tensor_shape(value)) for value in model.graph.output]
    print(f"{name:10} input={inputs} output={outputs} nodes={len(model.graph.node)}")

The batch size and image dimensions are dynamic in the source models. Dynamic shapes are useful for a general ONNX Runtime application, but each DEEPX compilation requires a concrete input shape.

## 4. Create fixed-shape ONNX models

The following table is the single source of truth for the fixed models. Shapes use NCHW order: batch, channels, height, width.

In [ ]:
FIXED_MODEL_SPECS = [
    {"name": "det_fixed",            "source": "det.onnx", "shape": [1, 3, 480, 480]},
    {"name": "cls_fixed",            "source": "cls.onnx", "shape": [1, 3, 48, 192]},
    {"name": "rec_fixed_ratio_2_5",  "source": "rec.onnx", "shape": [1, 3, 48, 120]},
    {"name": "rec_fixed_ratio_5",    "source": "rec.onnx", "shape": [1, 3, 48, 240]},
    {"name": "rec_fixed_ratio_10",   "source": "rec.onnx", "shape": [1, 3, 48, 480]},
    {"name": "rec_fixed_ratio_15",   "source": "rec.onnx", "shape": [1, 3, 48, 720]},
    {"name": "rec_fixed_ratio_25",   "source": "rec.onnx", "shape": [1, 3, 48, 1200]},
    {"name": "rec_fixed_ratio_35",   "source": "rec.onnx", "shape": [1, 3, 48, 1680]},
]

for spec in FIXED_MODEL_SPECS:
    print(f"{spec['name']:22} {spec['shape']}")

The notebook invokes ONNX Simplifier through the current kernel's Python interpreter. For example, the first conversion is equivalent to:

~~~bash
python -m onnxsim models/det.onnx models/det_fixed.onnx   --overwrite-input-shape x:1,3,480,480
~~~

Existing non-empty fixed models are checked and reused.

In [ ]:
import shlex

def fixed_model_path(spec) -> Path:
    return MODEL_DIR / f"{spec['name']}.onnx"

for spec in FIXED_MODEL_SPECS:
    source = MODEL_DIR / spec["source"]
    destination = fixed_model_path(spec)

    if destination.is_file() and destination.stat().st_size > 0:
        try:
            onnx.checker.check_model(str(destination))
            print(f"SKIP: {destination.name} already exists and is valid")
            continue
        except Exception:
            print(f"REBUILD: {destination.name} is not a valid ONNX model")
            destination.unlink()

    shape_argument = "x:" + ",".join(map(str, spec["shape"]))
    command = [
        sys.executable, "-m", "onnxsim",
        str(source), str(destination),
        "--overwrite-input-shape", shape_argument,
    ]
    print("\n" + shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)
    onnx.checker.check_model(str(destination))

### 4.1 Verify shapes and numerical equivalence

Structural validation alone is not enough. The next cells:

1. check every fixed ONNX model;
2. confirm its exact input shape;
3. compare ONNX Runtime outputs for detection, classification, and one representative recognition bucket.

All recognition buckets come from the same source graph. The ratio-3 model is used for the numerical recognition check to keep the tutorial validation quick.

In [ ]:
for spec in FIXED_MODEL_SPECS:
    path = fixed_model_path(spec)
    onnx.checker.check_model(str(path))
    model = onnx.load(path, load_external_data=False)
    actual_shape = tensor_shape(model.graph.input[0])
    if actual_shape != spec["shape"]:
        raise AssertionError(
            f"{path.name}: expected {spec['shape']}, got {actual_shape}"
        )
    print(f"PASS: {path.name:28} input={actual_shape}")

In [ ]:
import numpy as np
import onnxruntime as ort

NUMERICAL_CHECKS = [
    FIXED_MODEL_SPECS[0],  # detection
    FIXED_MODEL_SPECS[1],  # orientation classification
    FIXED_MODEL_SPECS[2],  # representative recognition bucket
]

rng = np.random.default_rng(2026)

for spec in NUMERICAL_CHECKS:
    source_path = MODEL_DIR / spec["source"]
    fixed_path = fixed_model_path(spec)
    sample = rng.random(spec["shape"], dtype=np.float32)

    source_session = ort.InferenceSession(
        str(source_path), providers=["CPUExecutionProvider"]
    )
    fixed_session = ort.InferenceSession(
        str(fixed_path), providers=["CPUExecutionProvider"]
    )
    source_output = source_session.run(None, {"x": sample})
    fixed_output = fixed_session.run(None, {"x": sample})

    if len(source_output) != len(fixed_output):
        raise AssertionError(f"Output count changed for {spec['name']}")

    max_error = max(
        float(np.max(np.abs(reference - candidate)))
        for reference, candidate in zip(source_output, fixed_output)
    )
    equivalent = all(
        np.allclose(reference, candidate, rtol=1e-4, atol=1e-5)
        for reference, candidate in zip(source_output, fixed_output)
    )
    print(f"{spec['name']:22} equivalent={equivalent} max_abs_error={max_error:.3e}")
    if not equivalent:
        raise AssertionError(f"Numerical output changed for {spec['name']}")

## 5. Create DX-COM calibration configurations

Calibration preprocessing must match application preprocessing. A mismatch in color order, scaling, normalization, or layout can reduce accuracy even when compilation succeeds.

| Model | Calibration images | Resize | Mean / standard deviation |
|---|---|---:|---|
| Detection | <code>det_dataset</code> | 480 × 480 | ImageNet values |
| Classification | <code>rec_dataset/ratio_5</code> | 192 × 48 | [0.5, 0.5, 0.5] |
| Recognition | Matching ratio bucket | Fixed width × 48 | [0.5, 0.5, 0.5] |

This tutorial retains the original DXQ-P0 enhanced scheme. For a product, compare its accuracy with a baseline compile on representative validation data instead of assuming that one quantization setting is always best.

In [ ]:
import json

DET_DATASET = TUTORIAL_DIR / "det_dataset"
REC_DATASET = TUTORIAL_DIR / "rec_dataset"

required_datasets = [
    DET_DATASET,
    REC_DATASET / "ratio_5",
    REC_DATASET / "ratio_15",
    REC_DATASET / "ratio_25",
]
for dataset in required_datasets:
    if not dataset.is_dir():
        raise FileNotFoundError(f"Calibration dataset was not found: {dataset}")

def image_count(directory: Path) -> int:
    extensions = {".jpeg", ".jpg", ".png"}
    return sum(
        path.is_file() and path.suffix.lower() in extensions
        for path in directory.iterdir()
    )

for dataset in required_datasets:
    count = image_count(dataset)
    if count == 0:
        raise RuntimeError(f"No calibration images were found in {dataset}")
    print(f"{dataset.relative_to(TUTORIAL_DIR)!s:28} {count:4} images")

In [ ]:
def preprocessing(width, height, mean, std):
    return [
        {"resize": {"width": width, "height": height}},
        {"convertColor": {"form": "BGR2RGB"}},
        {"div": {"x": 255}},
        {"normalize": {"mean": mean, "std": std}},
        {"transpose": {"axis": [2, 0, 1]}},
        {"expandDim": {"axis": 0}},
    ]

def calibration_config(shape, dataset, calibration_num, mean, std):
    _, _, height, width = shape
    return {
        "inputs": {"x": shape},
        "calibration_num": calibration_num,
        "calibration_method": "ema",
        "default_loader": {
            "dataset_path": str(dataset),
            "file_extensions": ["jpeg", "jpg", "png", "JPEG"],
            "preprocessings": preprocessing(width, height, mean, std),
        },
        "enhanced_scheme": {"DXQ-P0": {"alpha": 0.5}},
    }

REC_DATASET_BY_BUCKET = {
    3: REC_DATASET / "ratio_5",
    5: REC_DATASET / "ratio_5",
    10: REC_DATASET / "ratio_15",
    15: REC_DATASET / "ratio_15",
    25: REC_DATASET / "ratio_25",
    35: REC_DATASET / "ratio_25",
}

COMPILE_SPECS = []

for spec in FIXED_MODEL_SPECS:
    if spec["name"] == "det_fixed":
        config = calibration_config(
            spec["shape"], DET_DATASET, 100,
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225],
        )
    elif spec["name"] == "cls_fixed":
        config = calibration_config(
            spec["shape"], REC_DATASET / "ratio_5", 80,
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5],
        )
    else:
        bucket = int(spec["name"].rsplit("_", 1)[1])
        config = calibration_config(
            spec["shape"], REC_DATASET_BY_BUCKET[bucket], 80,
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5],
        )

    config_path = CONFIG_DIR / f"{spec['name']}.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
    COMPILE_SPECS.append({
        **spec,
        "onnx": fixed_model_path(spec),
        "config": config_path,
        "output_dir": OUTPUT_DIR / spec["name"],
    })
    print(f"WROTE: {config_path.relative_to(TUTORIAL_DIR)}")

Inspect at least one configuration before compiling. In particular, check NCHW input shape, calibration dataset, resize size, channel order, normalization, and transpose order.

In [ ]:
example_config = COMPILE_SPECS[0]["config"]
print(example_config.read_text(encoding="utf-8"))

## 6. Compile ONNX models to DXNN

Each model is written to its own directory. A valid existing DXNN file is skipped so that a repeated class does not spend time recompiling it.

The Python call is only used to display and execute the command reliably. For example, a compile is equivalent to:

~~~bash
source ~/dx-all-suite/dx-compiler/venv-dx-compiler-local/bin/activate
dxcom -m <fixed-model.onnx>       -c <calibration-config.json>       -o <output-directory>       --gen_log       --export_html
~~~

Unlike the original notebook, compiler output and failures are not redirected or converted into a successful cell.

In [ ]:
# Keep all entries for a complete OCR application.
# During a short class, you may select a smaller list only to demonstrate compilation.
SELECTED_COMPILES = [spec["name"] for spec in COMPILE_SPECS]
print("Models selected for compilation:")
for name in SELECTED_COMPILES:
    print(" -", name)

In [ ]:
def expected_dxnn(spec) -> Path:
    return spec["output_dir"] / f"{spec['name']}.dxnn"

def compile_model(spec) -> Path:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"SKIP: {artifact.relative_to(TUTORIAL_DIR)} already exists")
        return artifact

    spec["output_dir"].mkdir(parents=True, exist_ok=True)
    command = [
        str(DXCOM_PATH),
        "-m", str(spec["onnx"]),
        "-c", str(spec["config"]),
        "-o", str(spec["output_dir"]),
        "--gen_log",
        "--export_html",
    ]
    print("\nEquivalent terminal command:")
    print(f"source {shlex.quote(str(DX_COMPILER_VENV / 'bin' / 'activate'))}")
    print(shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)

    if not artifact.is_file() or artifact.stat().st_size == 0:
        raise FileNotFoundError(f"DX-COM did not create {artifact}")
    return artifact

COMPILED_MODELS = {}
for spec in COMPILE_SPECS:
    if spec["name"] in SELECTED_COMPILES:
        COMPILED_MODELS[spec["name"]] = compile_model(spec)

### 6.1 Verify compiled artifacts

A complete OCR application needs exactly eight DXNN files: detection, classification, and six recognition buckets. The cell fails early if any required file is missing.

In [ ]:
REQUIRED_DXNN_NAMES = [spec["name"] for spec in COMPILE_SPECS]
missing = []

for spec in COMPILE_SPECS:
    artifact = expected_dxnn(spec)
    if artifact.is_file() and artifact.stat().st_size > 0:
        print(f"PASS: {artifact.relative_to(TUTORIAL_DIR)!s:55} "
              f"{artifact.stat().st_size / 1_000_000:7.2f} MB")
    else:
        missing.append(artifact)

if missing:
    raise FileNotFoundError(
        "Missing DXNN files:\n" + "\n".join(f" - {path}" for path in missing)
    )

Use <code>dxparse</code> to inspect the three model roles. Recognition ratio 3 is representative; the other recognition files differ mainly in fixed input width.

In [ ]:
if not DXPARSE_PATH.is_file():
    raise FileNotFoundError(f"dxparse was not found: {DXPARSE_PATH}")

for model_name in ("det_fixed", "cls_fixed", "rec_fixed_ratio_3"):
    model_path = next(
        expected_dxnn(spec) for spec in COMPILE_SPECS if spec["name"] == model_name
    )
    command = [str(DXPARSE_PATH), "-m", str(model_path), "-v"]
    print("\n" + "=" * 72)
    print(shlex.join(command))
    subprocess.run(command, cwd=TUTORIAL_DIR, check=True)

## 7. Run the OCR application

The application performs the following work for each frame:

~~~text
Camera frame
    │
    ├─ Detection (once) ──► sorted text polygons
    │                          │
    │                          ├─ crop 0 ─► classify ─► rotate ─► ratio router ─► recognize
    │                          ├─ crop 1 ─► classify ─► rotate ─► ratio router ─► recognize
    │                          └─ ...
    │
    └────────────────────────────────────────────────────────────► draw text and polygons
~~~

The six recognition engines are loaded once. The router chooses one fixed-width model for each text crop.

### 7.1 Install the local DX-RT Python package

The package path comes from <code>config.json</code>. It works even when different users install <code>dx-all-suite</code> in different locations.

In [ ]:
dx_engine_install = [
    UV_PATH, "pip", "install",
    "--python", sys.executable,
    str(DX_ENGINE_PACKAGE_DIR),
]
print("Equivalent terminal command:")
print(shlex.join(dx_engine_install))
subprocess.run(dx_engine_install, check=True)

import cv2
import dx_engine

print("OpenCV   :", cv2.__version__)
print("dx_engine:", dx_engine.__file__)

### 7.2 Reviewed application safeguards

The supplied sample source is useful, but the following edge cases can make a demonstration misleading or unstable. The v6 runner below addresses them without editing <code>main.py</code> or files under <code>engine/</code>.

| Risk in the original flow | v6 behavior |
|---|---|
| Camera frame is cropped before checking whether capture succeeded | Check <code>ret</code> and <code>frame</code> first |
| No detected text causes division by zero in classification timing | Return an empty OCR result safely |
| Recognition removes low-confidence items, then text is matched by list position | Match text to polygons with <code>bbox_index</code> |
| OCR exceptions are silently converted to an empty result | Print the error and stop with a traceback |
| Camera and crop are hard-coded | Expose camera, width, height, FPS, and crop margin as CLI options |
| Model locations depend on the current directory | Resolve model paths from an explicit output directory |

### 7.3 Create the reviewed v6 runner

This cell creates <code>paddleocr_v6_runner.py</code> beside the notebook. It imports and reuses the existing preprocessing, post-processing, and drawing code; only control-flow safeguards are overridden.

In [ ]:
runner_source = r'''
import argparse
import sys
from pathlib import Path

import cv2
import numpy as np

from dx_engine import InferenceEngine as IE
from dx_engine import InferenceOption as IO

from engine.draw_utils import draw_with_poly_enhanced
from engine.paddleocr import PaddleOcr


class SafePaddleOcr(PaddleOcr):
    def __call__(self, image):
        detected, latency = self.detection_node(image)
        boxes = self.convert_boxes_to_quad_format(self.sorted_boxes(detected))
        self.detection_time_duration += latency * 1000
        self.ocr_run_count += 1

        if boxes.size == 0:
            return [], [], []

        crops = [self.get_rotate_crop_image(image, box) for box in boxes]
        boxes_as_lists = [box.tolist() for box in boxes]

        cls_results, latency = self.classification_node(crops)
        self.classification_time_duration += latency / len(crops) * 1000

        for index, (label, score) in enumerate(cls_results):
            if "180" in label and score > self.cls_thresh:
                crops[index] = cv2.rotate(crops[index], cv2.ROTATE_180)

        recognized, _, min_latency = self.recognition_node(
            image, boxes_as_lists, crops
        )
        if recognized:
            self.min_recognition_time_duration += min_latency * 1000

        return boxes_as_lists, crops, recognized


def draw_results(bgr_image, boxes, recognition_results):
    rgb_image = bgr_image[:, :, ::-1]
    text_by_box = {
        int(result["bbox_index"]): result["text"]
        for result in recognition_results
    }
    draw_items = [
        (
            [np.asarray(box).flatten()],
            text_by_box.get(index, ""),
            rgb_image.shape,
            rgb_image.shape,
        )
        for index, box in enumerate(boxes)
    ]
    return np.asarray(draw_with_poly_enhanced(rgb_image, draw_items))


def model_path(model_root, name):
    path = model_root / name / f"{name}.dxnn"
    if not path.is_file():
        raise FileNotFoundError(f"Required DXNN model was not found: {path}")
    return path


def open_camera(camera, width, height, fps):
    source = int(camera) if camera.isdecimal() else camera
    capture = cv2.VideoCapture(source, cv2.CAP_V4L2)
    if not capture.isOpened():
        raise RuntimeError(f"Could not open camera: {camera}")

    capture.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
    capture.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    capture.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
    capture.set(cv2.CAP_PROP_FPS, fps)
    return capture


def parse_args():
    parser = argparse.ArgumentParser(description="Reviewed PP-OCRv5 camera demo")
    parser.add_argument("--model-dir", type=Path, required=True)
    parser.add_argument("--camera", default="/dev/video0")
    parser.add_argument("--width", type=int, default=1920)
    parser.add_argument("--height", type=int, default=1080)
    parser.add_argument("--fps", type=float, default=5.0)
    parser.add_argument(
        "--crop-margin",
        type=int,
        default=0,
        help="Pixels removed from both the left and right sides",
    )
    return parser.parse_args()


def main():
    args = parse_args()
    model_root = args.model_dir.expanduser().resolve()
    option = IO().set_use_ort(True)

    detector = IE(str(model_path(model_root, "det_fixed")), option)
    classifier = IE(str(model_path(model_root, "cls_fixed")), option)
    recognizers = {
        ratio: IE(
            str(model_path(model_root, f"rec_fixed_ratio_{ratio}")),
            IO().set_use_ort(True),
        )
        for ratio in (3, 5, 10, 15, 25, 35)
    }
    worker = SafePaddleOcr(detector, classifier, recognizers)
    capture = open_camera(args.camera, args.width, args.height, args.fps)

    try:
        while True:
            ok, frame = capture.read()
            if not ok or frame is None:
                raise RuntimeError("Camera opened, but a frame could not be read.")

            if args.crop_margin:
                frame_width = frame.shape[1]
                if args.crop_margin * 2 >= frame_width:
                    raise ValueError(
                        f"crop-margin {args.crop_margin} is too large for "
                        f"the actual frame width {frame_width}"
                    )
                frame = frame[:, args.crop_margin : frame_width - args.crop_margin]

            boxes, _, recognized = worker(frame)
            rgb_result = draw_results(frame, boxes, recognized)
            cv2.imshow("PP-OCRv5", cv2.cvtColor(rgb_result, cv2.COLOR_RGB2BGR))

            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    except Exception as error:
        print(f"OCR demo stopped: {error}", file=sys.stderr)
        raise
    finally:
        capture.release()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()
'''

RUNNER_PATH = TUTORIAL_DIR / "paddleocr_v6_runner.py"
RUNNER_PATH.write_text(runner_source.lstrip(), encoding="utf-8")
subprocess.run([sys.executable, "-m", "py_compile", str(RUNNER_PATH)], check=True)
print(f"WROTE and validated: {RUNNER_PATH}")

### 7.4 Run the camera demo in a separate terminal

A camera window is an interactive GUI process. Run it in a JupyterLab terminal instead of blocking the notebook kernel.

Press **q** in the OpenCV window to exit.

The default command uses <code>/dev/video0</code> with no horizontal crop. Add <code>--crop-margin 440</code> only if the actual 1920-pixel frame needs the same center crop as the original demo.

In [ ]:
camera_command = [
    sys.executable,
    str(RUNNER_PATH),
    "--model-dir", str(OUTPUT_DIR),
    "--camera", "/dev/video0",
    "--width", "1920",
    "--height", "1080",
    "--fps", "5",
]

print("Run in a separate terminal:")
print(f"cd {shlex.quote(str(TUTORIAL_DIR))}")
print(shlex.join(camera_command))

<img src="assets/paddleocr-result.png" style="max-width: 980px;" alt="Expected PP-OCR result">

## 8. Accuracy and performance checklist

A successful compile does not guarantee acceptable OCR accuracy. Validate the whole pipeline with representative images.

| Check | Why it matters | Practical action |
|---|---|---|
| Detection resolution | Small text can disappear at 480 × 480 | Compare recall at the expected camera distance |
| Calibration coverage | Quantization follows calibration statistics | Include real lighting, fonts, blur, and backgrounds |
| Preprocessing parity | Color or normalization mismatch shifts model inputs | Keep compile and runtime preprocessing identical |
| Recognition bucket | Excessive resize or padding hurts characters | Measure text aspect-ratio distribution |
| Recognition confidence | A low threshold shows incorrect text; a high threshold drops text | Tune on a labeled validation set |
| Perspective and orientation | Camera documents are not always flat or upright | Add document orientation and unwarping when needed |
| End-to-end latency | One frame may contain many recognition crops | Report latency against text count, not only FPS |

Do not tune against a single attractive camera sample. Keep a fixed validation set and record detection recall, recognition accuracy, and end-to-end latency for every configuration change.

## 9. Troubleshooting

| Symptom | Check |
|---|---|
| <code>ROOT_PATH is not set</code> | Start JupyterLab with <code>./run-jupyter-lab.sh</code> |
| <code>dxcom</code> path is missing | Complete the DX-COM installation and check <code>config.json</code> |
| Downloaded file is invalid | Delete only the named model and rerun Section 3 |
| Fixed model shape is wrong | Delete that fixed ONNX file and rerun Section 4 |
| Compilation fails | Read the visible DX-COM output and the model output directory's log |
| <code>dx_engine</code> import fails | Rerun Section 7.1 with the current notebook kernel |
| Camera cannot open | Check permissions and the selected <code>/dev/video*</code> device |
| Qt/display error | Run from a graphical JupyterLab terminal with a valid <code>DISPLAY</code> |
| No text is shown | Check detector output, crop lighting/focus, recognition confidence, and model paths |

## Summary

In this tutorial, you:

- mapped PP-OCRv5 into detection, orientation classification, and recognition stages;
- converted dynamic ONNX inputs into eight validated fixed-shape models;
- checked representative ONNX outputs for numerical equivalence;
- generated consistent calibration configurations from one model table;
- compiled models with visible, fail-fast DX-COM commands;
- verified the complete set of DXNN artifacts;
- created a safer camera runner that handles empty detections, camera failures, text-to-box alignment, configurable capture settings, and model paths;
- learned which data and preprocessing checks matter before measuring OCR accuracy.

The most important release rule is to validate the complete application on representative documents. Model compilation, runtime execution, and visual inspection are necessary checks, but they are not substitutes for a labeled accuracy evaluation.